## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

## Load the Dataset

In [11]:
df = pd.read_csv("../data/clean_data.csv")
df.head()

,match_result,home_club,away_club,fixture_date,competition,competition_code,cup_game,home_manager_id,away_manager_id,home_club_recent_fixture_date_1,...,away_club_recent_competition_code_1,away_club_recent_competition_code_2,away_club_recent_competition_code_3,away_club_recent_competition_code_4,away_club_recent_competition_code_5,away_club_recent_competition_code_6,away_club_recent_competition_code_7,away_club_recent_competition_code_8,away_club_recent_competition_code_9,away_club_recent_competition_code_10
0,away,Newell's Old Boys,River Plate,2019-12-01 00:45:00,Superliga,636,False,468196.0,468200.0,2019-11-26 00:10:00,...,1122.0,642.0,636.0,636.0,636.0,1122.0,636.0,642.0,636.0,1122.0
1,home,Real Estelí,Deportivo Las Sabanas,2019-12-01 01:00:00,Primera Division,752,False,516788.0,22169161.0,2019-11-27 21:00:00,...,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0
2,draw,UPNFM,Marathón,2019-12-01 01:00:00,Liga Nacional,734,False,2510608.0,456313.0,2019-11-28 01:15:00,...,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0
3,away,León,Morelia,2019-12-01 01:00:00,Liga MX,743,False,1552508.0,465797.0,2019-11-28 01:00:00,...,743.0,743.0,743.0,743.0,743.0,743.0,743.0,743.0,746.0,743.0
4,home,Cobán Imperial,Iztapa,2019-12-01 01:00:00,Liga Nacional,705,False,429958.0,426870.0,2019-11-27 18:00:00,...,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0


## Time-Based Feature Extraction
The `fixture_date` column is used to extract time-based features such as year, month, and day of the week. 

Before extracting these features, the dates are standardized to UTC. Crucially, the dataset is then sorted chronologically. This completely prevents future data leakage when training our models, ensuring that past data strictly predicts future data.

In [12]:
# Convert to datetime and standardize to UTC
df['fixture_date'] = pd.to_datetime(df['fixture_date'], utc=True)

# Sort chronologically to prevent data leakage
df = df.sort_values('fixture_date').reset_index(drop=True)

# Extract time-based features
df['fixture_year'] = df['fixture_date'].dt.year
df['fixture_month'] = df['fixture_date'].dt.month
df['fixture_day_of_week'] = df['fixture_date'].dt.dayofweek

# Create a season indicator (differentiating summer break months)
df['is_summer_break'] = df['fixture_month'].isin([6, 7]).astype(int)

## Schedule Congestion and Fatigue Indicators
Machine learning models cannot process raw timestamp strings. Furthermore, the absolute date of a previous match is less important than the relative time elapsed. 

Here, we convert the 10 historical fixture dates into a "days since match" numerical feature for both teams. This mathematically represents schedule congestion and potential team fatigue. Afterward, the original raw string columns are dropped to keep the dataset model-ready.

In [ ]:
def calculate_rest_days(df, team_type):
    for i in range(1, 11):
        hist_date_col = f"{team_type}_club_recent_fixture_date_{i}"
        rest_days_col = f"{team_type}_days_since_match_{i}"
        
        df[hist_date_col] = pd.to_datetime(df[hist_date_col], errors='coerce', utc=True)
        
        df[rest_days_col] = (df['fixture_date'] - df[hist_date_col]).dt.days
        
        # If a team hasn't played 10 previous matches, fill the missing rest days with a logical default (90 days)
        df[rest_days_col] = df[rest_days_col].fillna(90)
        
        # Drop the original unreadable string column
        df.drop(columns=[hist_date_col], inplace=True)
        
    return df

# Apply the fatigue calculation to both home and away teams
df = calculate_rest_days(df, "home")
df = calculate_rest_days(df, "away")

df.drop(columns=['fixture_date'], inplace=True)

,match_result,home_club,away_club,competition,competition_code,cup_game,home_manager_id,away_manager_id,home_club_recent_played_at_home_1,home_club_recent_played_at_home_2,...,away_days_since_match_1,away_days_since_match_2,away_days_since_match_3,away_days_since_match_4,away_days_since_match_5,away_days_since_match_6,away_days_since_match_7,away_days_since_match_8,away_days_since_match_9,away_days_since_match_10
0,away,Newell's Old Boys,River Plate,Superliga,636,False,468196.0,468200.0,0.0,1.0,...,7,16,20,28,32,39,43,50,55,60
1,home,Real Estelí,Deportivo Las Sabanas,Primera Division,752,False,516788.0,22169161.0,1.0,0.0,...,3,7,21,28,34,41,45,63,71,77
2,draw,UPNFM,Marathón,Liga Nacional,734,False,2510608.0,456313.0,0.0,1.0,...,9,20,35,41,45,49,56,63,69,73
3,away,León,Morelia,Liga MX,743,False,1552508.0,465797.0,0.0,0.0,...,3,7,22,26,30,36,42,56,59,65
4,home,Cobán Imperial,Iztapa,Liga Nacional,705,False,429958.0,426870.0,0.0,1.0,...,3,6,20,28,35,38,41,56,59,62


## Match Context Differentiation
Cup matches often have different patterns than regular competition matches. We need to ensure the `cup_game` flag is properly transformed into a strict numerical format.

In [5]:
df['cup_game'] = df['cup_game'].astype(int)

## Historical Form and Performance Indicators
To effectively describe recent club performance and consistency, we calculate aggregate statistics from the last 10 matches. We will engineer the mean goals scored, mean goals conceded, and mean team ratings over the recent rolling window for both the home and away clubs.

In [6]:
def calculate_recent_stats(df, team_type):
    goals_scored_cols = [f"{team_type}_club_recent_goals_scored_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_goals_scored'] = df[goals_scored_cols].mean(axis=1)

    goals_conceded_cols = [f"{team_type}_club_recent_goals_conceded_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_goals_conceded'] = df[goals_conceded_cols].mean(axis=1)

    rating_cols = [f"{team_type}_club_recent_team_rating_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_team_rating'] = df[rating_cols].mean(axis=1)

    return df

# Apply aggregations to both home and away teams
df = calculate_recent_stats(df, "home")
df = calculate_recent_stats(df, "away")

## Categorical Data Transformation
Categorical attributes such as club names and competition names must be transformed into a numerical format for the machine learning algorithms. We apply Label Encoding to these features, as well as to our 3-class target variable (`match_result`). Manager IDs are also strictly cast to numerical floats.

In [7]:
le = LabelEncoder()

# Categorical columns to encode
cat_cols = ['home_club', 'away_club', 'competition']

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# Encode target variable (match_result: home, away, draw) into 0, 1, 2
df['match_result'] = le.fit_transform(df['match_result'].astype(str))

# Ensure manager IDs are numerical
df['home_manager_id'] = df['home_manager_id'].astype(float)
df['away_manager_id'] = df['away_manager_id'].astype(float)

## Save Engineered Data
The engineered dataframe is saved and ready to be loaded into the modeling pipeline.

In [8]:
df.to_csv("../data/engineered_data.csv", index=False)
print(f"Feature engineering complete. Data matrix shape: {df.shape}")

Feature engineering complete. Data matrix shape: (110921, 198)
